# Indexing several materials (1c/3)

Index and refine the grains of several materials (phases, substrate...) present in the same Laue patterns. Each image is indexed by a **sequence of steps** (one material per step, with its own parameters); the spots of a kept grain are removed for the next grains, so a spot of a .cor file belongs to one grain only.

| STRATEGY | in each image | when |
|---|---|---|
| `'ordered'` (default) | the steps one after the other: each one indexes grains of its material (up to `nbGrainstoFind`) until a grain is not kept; `next` of the step decides to go on (`always`, `if_found`, `if_not_found`) | the order of the materials is known (e.g. substrate, then main phase, then minor phases) |
| `'round_robin'` | rounds of one grain of each material | materials with similar spots numbers, no priority |
| `'best_of'` | in each round, one grain is searched for each material and only the best one is kept (`BEST_OF_CRITERION`: most indexed spots or highest matching rate) | unknown order; more robust (more computing time) |

A grain is **kept** if it fulfils the `accept` criteria of its step (min nb of indexed spots, min matching rate, max pixel deviation) and indexes at least `MIN_GAIN` spots (for `'round_robin'`: the round, with all materials tried, must add `MIN_GAIN` spots). Indexing of an image stops at `MAX_GRAINS` grains (all materials) and `MAX_GRAINS_PER_MATERIAL`.

Results (folder `RESULTS_ROOT`): `fitfiles_<folder>/<prefix>####_g0.fit, _g1.fit ...`, one folder per material (or per `folder` of a step, e.g. `substrate`).

**This notebook is part of the LaueTools package** (https://github.com/BM32ESRF/lauetools). Tested with python 3.12. Module: `LaueTools/indexing_sequence.py` (documentation of the `sequence` section of the configuration).

# SETTINGS

In [ ]:
CONFIG = 'configs/322812_Al_Al2Cu.yaml'           # configuration of the experiment (folders, default indexing parameters)
NEW_CONFIG = 'configs/322812_multimaterials.yaml'  # configuration written by this notebook (with the sequence of indexing)
RESULTS_ROOT = None    # folder of the fitfiles_<...> folders. None: <parent of fit_folder of CONFIG>/sequence_<name of NEW_CONFIG>

STRATEGY = 'ordered'              # 'ordered', 'round_robin' or 'best_of'
MAX_GRAINS = 6                    # max nb of grains per image (all materials). None: no limit
MAX_GRAINS_PER_MATERIAL = 3       # max nb of grains of each material per image. None: no limit
MIN_GAIN = 8                      # min nb of indexed spots added by a kept grain ('round_robin': by a round)
BEST_OF_CRITERION = 'nb_indexed'  # 'best_of': 'nb_indexed' or 'matching_rate'

# sequence of steps (order = order of 'ordered' and 'round_robin'). Keys of a step (all optional but key_material):
# name, key_material, folder (fitfiles_<folder>, default key_material), nbGrainstoFind, e_min, e_max, e_max_MR,
# dict_indexrefine (changes of indexing parameters of CONFIG), usepreviousUB (names of ub_matrices of CONFIG tested first),
# skipindexing, accept {min_nb_indexed, min_matching_rate, max_pixdev}, remove_spots, write_fitfile, next
STEPS = [
    {'name': 'substrate', 'key_material': 'Al2O3', 'folder': 'substrate', 'nbGrainstoFind': 1,
     'usepreviousUB': ['Al2O3_1'],
     'dict_indexrefine': {'central spots indices': 'to9', 'MAXINDEXPROBED': 30, 'nlutmax': 4},
     'accept': {'min_nb_indexed': 20, 'min_matching_rate': 20}},
    {'name': 'matrix', 'key_material': 'Al', 'nbGrainstoFind': 2, 'e_max_MR': 19, 'e_max': 22,
     'dict_indexrefine': {'central spots indices': 'to9', 'MAXINDEXPROBED': 25, 'nlutmax': 4},
     'accept': {'min_nb_indexed': 10, 'min_matching_rate': 15, 'max_pixdev': 1.5}},
    # large tetragonal cell: many simulated spots, low matching rate -> criteria on nb of spots and pixel deviation
    {'name': 'precipitate', 'key_material': 'Al2Cu', 'nbGrainstoFind': 2,
     'dict_indexrefine': {'central spots indices': 'to19', 'MAXINDEXPROBED': 21, 'nlutmax': 3},
     'accept': {'min_nb_indexed': 20, 'min_matching_rate': 3, 'max_pixdev': 1.5}},
]

COMPARE_STRATEGIES = True   # test: the 3 strategies on the test image
IMAGE_RANGE = None     # [first, last] image indices of the batch. None: selection of CONFIG (all files by default)
NB_CPUS = None         # None: nb_cpus of CONFIG (or all cpus)
RUN_BATCH = True
EXECUTION = 'local'    # 'local' or 'slurm'
SAVE_RESULTS = True

# IMPORTS

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"   # before numpy import: 1 thread per process (images are dispatched on cpus)
import copy
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import ipympl  # interactive plots (zoom, hover) in JupyterLab / VS Code
    %matplotlib widget
except ImportError:
    %matplotlib inline

import LaueTools.indexing_batch as IB
import LaueTools.indexing_sequence as IS
import LaueTools.map_workflow as MW
import LaueTools.IOLaueTools as IOLT
import LaueTools.generaltools as GT

print('LaueTools from', Path(IB.__file__).parent)
print(f'nb of cpus available: {os.cpu_count()}')

# LOAD CONFIGURATION and write the new configuration with the sequence

The sequence is written in `NEW_CONFIG` (section `sequence`); it can also be edited there, then load it with `cfg_seq = IB.load_config(NEW_CONFIG)`.

In [ ]:
cfg = IB.load_config(CONFIG)
mapdims = cfg['scan']['mapdims']
results_root = (Path(RESULTS_ROOT) if RESULTS_ROOT
                else Path(cfg['output']['fit_folder']).parent / f'sequence_{Path(NEW_CONFIG).stem}')
sequence = {'strategy': STRATEGY, 'max_grains': MAX_GRAINS, 'max_grains_per_material': MAX_GRAINS_PER_MATERIAL,
            'min_gain_indexed_spots': MIN_GAIN, 'best_of_criterion': BEST_OF_CRITERION,
            'folders_root': str(results_root), 'steps': STEPS}
changes = {} if IMAGE_RANGE is None else {'image_range': list(IMAGE_RANGE)}
cfg_seq = IB.save_config(cfg, NEW_CONFIG, overwrite=True, sequence=sequence, fit_folder=str(results_root),
                         results_folder=str(results_root), results_stamp='sequence', **changes)
config_ok = IB.check_config(cfg_seq)
for st in IS.normalize_sequence(cfg_seq)['steps']:
    print(f"step {st['name']:<12s} {st['key_material']:<8s} -> fitfiles_{st['folder']:<10s} max {st['nbGrainstoFind']} grain(s), "
          f"e_max_MR {st['e_max_MR']} e_max {st['e_max']} keV, accept {st['accept']}")
print('results in', results_root)

# TEST on 1 image

All the attempts are printed: kept grains, grains not found, rejected by the `accept` criteria, gain too small, or not the best (`best_of`). The .fit files of this image are written in the result folders.

In [ ]:
TEST_IMAGE = cfg_seq['run']['test_image_index']
testcor = IB.corfile_path(cfg_seq, TEST_IMAGE)
params_test = IB.params_from_config(cfg_seq, useinternalmultiprocessing=True, ignorefitfileresults=True)
t0 = time.time()
res = IB.index_refine(testcor, params_test)
IS.print_records(res)
print(f'elapsed time {time.time() - t0:.1f} s')
IS.plot_records(res, testcor)
plt.show()

## [OPTION] compare the 3 strategies on the test image

(no .fit file written)

In [ ]:
if COMPARE_STRATEGIES:
    rows = []
    for strategy in ('ordered', 'round_robin', 'best_of'):
        p = copy.deepcopy(params_test)
        p.sequence['strategy'] = strategy
        p.writefitfile = False
        t0 = time.time()
        r = IB.index_refine(testcor, p)
        recs = IS.records_of(r)
        rows.append((strategy, len(recs), sum(x['nb_indexed'] for x in recs),
                     ', '.join(f"{x['key_material']}:{x['nb_indexed']}" for x in recs), time.time() - t0))
    print(f"{'strategy':<12s} {'grains':>6s} {'indexed spots':>13s}  {'time (s)':>8s}  grains (material:nb spots)")
    for strategy, ngrains, nspots, detail, dt in rows:
        print(f'{strategy:<12s} {ngrains:6d} {nspots:13d}  {dt:8.1f}  {detail}')

# BATCH: all .cor files of the selection

- set of files: section `selection` of the new configuration (`IMAGE_RANGE` in SETTINGS, or all files)
- `EXECUTION = 'slurm'` (SETTINGS): job(s) on the ESRF cluster with exactly these parameters (cells below)
- run again: the .fit files of the analysed images are replaced (`run: ignorefitfileresults: false` in the configuration: only images without results)

In [ ]:
params = IB.params_from_config(cfg_seq)
listfiles = IB.select_corfiles(cfg_seq)
print(f'{len(listfiles)} .cor files to analyse, from {Path(listfiles[0]).name} to {Path(listfiles[-1]).name}')
nb_cpus = NB_CPUS or cfg_seq['run']['nb_cpus'] or IB.available_cpus()

allresults = None
if RUN_BATCH and config_ok and EXECUTION == 'local':
    t0 = time.time()
    allresults = IB.run_multiprocessing(listfiles, params, nb_cpus=nb_cpus)
    GT.printgreen(f'Done in {time.time() - t0:.0f} s')

## [OPTION] BATCH on SLURM (ESRF cluster)

With `EXECUTION = 'slurm'` in SETTINGS: the job folder (parameters, script, logs, results) is written in `slurm_jobs/` next to the results. Submit from this notebook in a jupyter-slurm session (`SUBMIT = True`), or with the printed `sbatch` command.

In [ ]:
SLURM_MACHINE = 'magnifix'   # 'auto', 'magnifix', 'magnifix2', 'magnifix3', 'hpc6', 'hpc7', 'hpc8', 'nice'
SLURM_TIME = '1:00:00'       # time limit hh:mm:ss
SLURM_CPUS = 96              # cpus per job (1 node)
NCHUNKS = 1                  # nb of jobs (job array)
SUBMIT = False               # True: sbatch now (jupyter-slurm session)

job = None
if EXECUTION == 'slurm' and config_ok:
    machine, ncpus = IB.choose_slurm_machine(SLURM_CPUS) if SLURM_MACHINE == 'auto' else (SLURM_MACHINE, SLURM_CPUS)
    job = IB.prepare_slurm_job(cfg_seq, params, listfiles, machine=machine, nb_cpus=ncpus, time=SLURM_TIME,
                               nchunks=NCHUNKS, jobs_folder=results_root / 'slurm_jobs')
    if SUBMIT:
        IB.submit_slurm_job(job)

In [ ]:
# job state (run again to update), then results of the finished job(s)
if EXECUTION == 'slurm' and job is not None:
    IB.slurm_job_status(job)
    allresults = IB.load_slurm_results(job)

## SUMMARY of the batch

- phase map: material (folder) of the grain with most indexed spots in each image
- nb of grains of each folder, total nb of grains

In [ ]:
if allresults is None:
    raise RuntimeError('no batch results: set RUN_BATCH = True (and check the configuration)')
maps = IS.summarize_sequence(allresults, mapdims)
folders = [k for k in maps if k != 'nb_grains_total']
phases, labels = IS.phase_map(allresults, mapdims)
print('phases:', {k: lab for k, lab in enumerate(labels)}, '(-1: no grain)')

panels = [(phases, 'phase: ' + ', '.join(f'{k}={lab}' for k, lab in enumerate(labels)), 'tab10', -1, 9),
          (maps['nb_grains_total'], 'nb of grains (all materials)', 'viridis', None, None)]
panels += [(maps[f]['nb_grains'], f'fitfiles_{f}: nb of grains', 'viridis', None, None) for f in folders]
fig, axes = plt.subplots(1, len(panels), figsize=(4.4 * len(panels), 4.2), squeeze=False)
for ax, (data, title, cmap, vmin, vmax) in zip(axes[0], panels):
    MW.plot_map(data, cfg_seq, title, ax=ax, cmap=cmap, vmin=vmin, vmax=vmax)
fig.tight_layout()
plt.show()

## SAVE results and configurations of each folder

- `allresults_sequence.pickle` in the results folder (reload: `IB.load_allresults(path)`, kept grains of image `res`: `IS.records_of(res)`)
- one configuration per folder of .fit files (`<NEW_CONFIG>_<folder>.yaml`: its material and `fit_folder`, no sequence) to use as `CONFIG` in `2_postprocess_maps.ipynb` and `3_segmentation_grains.ipynb`

In [ ]:
if SAVE_RESULTS and allresults is not None:
    IB.save_allresults(cfg_seq, params, allresults, IB.summarize_results(allresults, mapdims), overwrite=True,
                       nb_cpus=nb_cpus)
    for folder in IS.step_folders(params.sequence):
        cfg_folder = IS.folder_config(cfg_seq, folder)
        print(f"fitfiles_{folder}: CONFIG = '{cfg_folder['configfile']}'")

**Next step:** `2_postprocess_maps.ipynb` and `3_segmentation_grains.ipynb` with `CONFIG` = the configuration of the folder (material) to analyse (printed above)

## Export of this notebook (HTML, PDF)

Save the notebook first (Ctrl+S): the export reads the saved `.ipynb` file, with its outputs. Files `<notebook name>.html` / `.pdf`. Same as in a terminal: `lauetools-export-notebook <notebook>.ipynb`

In [ ]:
from LaueTools.notebook_export import export_this_notebook

EXPORT_TO = ['html', 'pdf']
EXPORT_FOLDER = None    # None: folder of the notebook
export_this_notebook(to=EXPORT_TO, outdir=EXPORT_FOLDER);